In [ ]:
# ------------------------------------------------------------
# CELL 0 — mount Google Drive
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ------------------------------------------------------------
# CELL 1 — install any missing packages (skip if already installed)
# ------------------------------------------------------------
!pip install shap timm --quiet

In [ ]:
# ------------------------------------------------------------
# CELL 2 — clone the repo (code only, checkpoints are gitignored)
# ------------------------------------------------------------
!git clone https://github.com/Jmanav/SkinSure.git /content/SkinSure

Cloning into '/content/SkinSure'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 142 (delta 50), reused 142 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 226.99 KiB | 5.54 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [ ]:
# ------------------------------------------------------------
# CELL 3 — cd into repo + fix torch.load for PyTorch 2.6
# + reduce CUDA memory fragmentation (recommended by the error msg)
# ------------------------------------------------------------
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.chdir('/content/SkinSure')

import functools
import torch
torch.load = functools.partial(torch.load, weights_only=False)

In [ ]:
# ------------------------------------------------------------
# CELL 4 — paths (OUTPUT now nested inside XAI/SHAP/)
# ------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/IPD/XAI"

BACKGROUND_DIR = f"{DRIVE_ROOT}/SHAP/background"
TEST_DIR = f"{DRIVE_ROOT}/SHAP/test"

OUTPUT_DIR = f"{DRIVE_ROOT}/SHAP/SHAP_newfolder_ensemble_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINTS_DIR = "/content/drive/MyDrive/IPD/weights_skinsure"
EFFICIENTNET_CKPT = f"{CHECKPOINTS_DIR}/efficientnet_b1_5cls_skin.pth"
CONVNEXT_CKPT = f"{CHECKPOINTS_DIR}/convnext_tiny_5cls_skin.pth"
SWIN_CKPT = f"{CHECKPOINTS_DIR}/swin_tiny_5cls_skin.pth"

print("Background folder:", BACKGROUND_DIR)
print("Test folder:", TEST_DIR)
print("Output folder:", OUTPUT_DIR)

Background folder: /content/drive/MyDrive/IPD/XAI/SHAP/background
Test folder: /content/drive/MyDrive/IPD/XAI/SHAP/test
Output folder: /content/drive/MyDrive/IPD/XAI/SHAP/SHAP_newfolder_ensemble_results


In [ ]:
# ------------------------------------------------------------
# CELL 5 — build the ensemble wrapper
# ------------------------------------------------------------
import torch.nn as nn
from src.models.efficientnet import load_efficientnet_b1
from src.models.convnext import load_convnext_tiny
from src.models.swin import load_swin_tiny

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EnsembleWrapper(nn.Module):
    def __init__(self, efficientnet_path, convnext_path, swin_path, device):
        super().__init__()
        self.device = device
        self.efficientnet, classes_a = load_efficientnet_b1(efficientnet_path, device)
        self.convnext, classes_b = load_convnext_tiny(convnext_path, device)
        self.swin, classes_c = load_swin_tiny(swin_path, device)
        assert classes_a == classes_b == classes_c, "Class order mismatch between checkpoints!"
        self.classes = classes_a
        self.efficientnet.eval()
        self.convnext.eval()
        self.swin.eval()

    def forward(self, x):
        prob_a = torch.softmax(self.efficientnet(x), dim=1)
        prob_b = torch.softmax(self.convnext(x), dim=1)
        prob_c = torch.softmax(self.swin(x), dim=1)
        return (prob_a + prob_b + prob_c) / 3.0

ensemble = EnsembleWrapper(EFFICIENTNET_CKPT, CONVNEXT_CKPT, SWIN_CKPT, DEVICE).to(DEVICE)
print("Ensemble classes:", ensemble.classes)

✓ EfficientNet-B1 | val=87.69% test=85.11% | mel_sens=98.6%
✓ ConvNeXt-Tiny | val=75.57% test=71.35% | mel_sens=98.6%
✓ Swin-Tiny | val=88.35% test=85.39% | mel_sens=99.1%
Ensemble classes: ['Basal_Cell_Carcinoma', 'Eczema', 'Keratosis', 'Melanoma', 'Tinea']


In [ ]:
# ------------------------------------------------------------
# CELL 6 — load the SAME background/test images used for the
# individual-model SHAP runs (fair comparison, same preprocessing)
# ------------------------------------------------------------
from PIL import Image
from src.preprocessing.inference_transform import preprocess_image
import numpy as np

def load_images_from_folder(folder_path, classes):
    tensors = []
    meta = []
    for class_name in classes:
        class_dir = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_dir):
            print(f"WARNING: no folder found for class '{class_name}' at {class_dir}")
            continue
        for fname in sorted(os.listdir(class_dir)):
            fpath = os.path.join(class_dir, fname)
            try:
                img = Image.open(fpath).convert("RGB")
            except Exception as e:
                print(f"Skipping {fpath}: {e}")
                continue
            tensor = preprocess_image(img)
            if tensor.dim() == 4:          # already has a batch dimension — remove it
                tensor = tensor.squeeze(0)
            tensors.append(tensor)
            meta.append((fname, class_name))
    if not tensors:
        raise RuntimeError(f"No images loaded from {folder_path} — check the path and class folder names.")
    return torch.stack(tensors).to(DEVICE), meta

background_tensor, background_meta = load_images_from_folder(BACKGROUND_DIR, ensemble.classes)
test_tensor, test_meta = load_images_from_folder(TEST_DIR, ensemble.classes)

print(f"Loaded {background_tensor.shape[0]} background images")
print(f"Loaded {test_tensor.shape[0]} test images")
print(f"Background tensor shape: {background_tensor.shape}")
print(f"Test tensor shape: {test_tensor.shape}")

Loaded 40 background images
Loaded 15 test images
Background tensor shape: torch.Size([40, 3, 224, 224])
Test tensor shape: torch.Size([15, 3, 224, 224])


In [ ]:
# ------------------------------------------------------------
# CELL 7 — run SHAP on the ensemble, save 3-panel figures matching
# the diverging red/blue style (True/Pred labels, signed heatmap,
# overlay). Empties the output folder first, then saves each result
# into its own per-class subfolder.
# ------------------------------------------------------------
import gc
import shutil
import numpy as np
import shap
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def unnormalize(tensor_img):
    img = tensor_img.detach().cpu().numpy().transpose(1, 2, 0)
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

# Empty the output folder (fresh start) before saving new results
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output folder cleared:", OUTPUT_DIR)

torch.cuda.empty_cache()
gc.collect()

explainer = shap.GradientExplainer(ensemble, background_tensor, local_smoothing=0.5, batch_size=1)

for i in range(test_tensor.shape[0]):
    fname, true_class = test_meta[i]
    single_img = test_tensor[i:i+1]

    # Get the ensemble's predicted class for the True/Pred title
    with torch.no_grad():
        probs = ensemble(single_img)
        pred_idx = torch.argmax(probs, dim=1).item()
        pred_class = ensemble.classes[pred_idx]

    result = explainer.shap_values(single_img, ranked_outputs=1, nsamples=150)

    if isinstance(result, tuple):
        shap_values, indexes = result
    else:
        shap_values, indexes = result, None

    if isinstance(shap_values, list):
        sv = shap_values[0]
    else:
        sv = shap_values

    sv = np.array(sv)
    sv = sv.reshape(sv.shape[1:4]) if sv.ndim == 5 else sv[0]

    # KEEP THE SIGN this time (don't take abs) — this is what gives the
    # red = pushes toward prediction / blue = pushes away style
    heatmap = np.sum(sv, axis=0)
    heatmap_blurred = gaussian_filter(heatmap, sigma=3)

    # Symmetric color range around 0, same as the reference images
    vmax = np.abs(heatmap_blurred).max()
    vmin = -vmax

    original_img = unnormalize(single_img[0])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original_img)
    axes[0].set_title(f"True: {true_class}\nPred: {pred_class}", loc="left", fontsize=11)
    axes[0].axis("off")

    im = axes[1].imshow(heatmap_blurred, cmap="bwr", vmin=vmin, vmax=vmax)
    axes[1].set_title("SHAP (Ensemble)")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(original_img)
    axes[2].imshow(heatmap_blurred, cmap="bwr", vmin=vmin, vmax=vmax, alpha=0.5)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()

    # Per-class subfolder — e.g. OUTPUT_DIR/Melanoma/xxx_ensemble_shap.png
    class_dir = os.path.join(OUTPUT_DIR, true_class)
    os.makedirs(class_dir, exist_ok=True)

    out_path = os.path.join(class_dir, f"{fname}_ensemble_shap.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    np.save(os.path.join(class_dir, f"{fname}_ensemble_shap.npy"), heatmap)

    print(f"  [{i+1}/{test_tensor.shape[0]}] Saved: {true_class}/{fname} (pred: {pred_class})")

    del result, shap_values, sv, heatmap, heatmap_blurred, original_img, fig
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nDone! All ensemble SHAP results saved to: {OUTPUT_DIR} (organized by class subfolder)")


Output folder cleared: /content/drive/MyDrive/IPD/XAI/SHAP/SHAP_newfolder_ensemble_results
  [1/15] Saved: Basal_Cell_Carcinoma/ISIC_0065249.jpg (pred: Basal_Cell_Carcinoma)
  [2/15] Saved: Basal_Cell_Carcinoma/ISIC_0071603.jpg (pred: Basal_Cell_Carcinoma)
  [3/15] Saved: Basal_Cell_Carcinoma/ISIC_0073081.jpg (pred: Basal_Cell_Carcinoma)
  [4/15] Saved: Eczema/6_29.jpg (pred: Eczema)
  [5/15] Saved: Eczema/IMG20240403112731.jpg (pred: Eczema)
  [6/15] Saved: Eczema/IMG_20230531_124822_1.jpg (pred: Tinea)
  [7/15] Saved: Keratosis/ISIC_0062717.jpg (pred: Basal_Cell_Carcinoma)
  [8/15] Saved: Keratosis/ISIC_0071952.jpg (pred: Basal_Cell_Carcinoma)
  [9/15] Saved: Keratosis/v-epidermal-cyst-106.jpg (pred: Keratosis)
  [10/15] Saved: Melanoma/3635.jpg (pred: Melanoma)
  [11/15] Saved: Melanoma/ISIC_6895778.jpg (pred: Keratosis)
  [12/15] Saved: Melanoma/ISIC_7303761.jpg (pred: Keratosis)
  [13/15] Saved: Tinea/IMG_20230214_120109_1.jpg (pred: Tinea)
  [14/15] Saved: Tinea/IMG_20230705_1225

In [ ]:
# ------------------------------------------------------------
# CELL 8 — AOPC Deletion Test on the ensemble (same methodology as
# your individual-model AOPC test: progressively blank the most
# important pixels, track how much the prediction confidence drops)
# ------------------------------------------------------------
import glob

DELETION_STEPS = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]  # % of pixels blanked

def run_deletion_test(image_tensor, heatmap, ensemble, pred_idx, device):
    """
    Ranks pixels by |importance| (highest first), then progressively
    blanks that percentage of the image (sets to 0 in normalized space,
    equivalent to a neutral gray fill) and records the model's
    confidence in the ORIGINAL predicted class at each step.
    """
    H, W = heatmap.shape
    importance_flat = np.abs(heatmap).flatten()
    # Indices sorted by importance, descending (most important first)
    sorted_idx = np.argsort(-importance_flat)

    confidences = []
    img = image_tensor.clone()

    for pct in DELETION_STEPS:
        k = int(len(sorted_idx) * pct / 100)
        modified = image_tensor.clone()

        if k > 0:
            top_k_idx = sorted_idx[:k]
            rows = top_k_idx // W
            cols = top_k_idx % W
            # Zero out these pixels across all 3 color channels
            modified[:, :, rows, cols] = 0.0

        with torch.no_grad():
            probs = ensemble(modified.to(device))
            conf = probs[0, pred_idx].item()
        confidences.append(conf)

    return confidences


# Empty/create a folder for AOPC results
AOPC_OUTPUT_DIR = f"{DRIVE_ROOT}/SHAP/SHAP_newfolder_ensemble_results/AOPC_results"
os.makedirs(AOPC_OUTPUT_DIR, exist_ok=True)

all_curves = []
per_class_aopc = {}

for i in range(test_tensor.shape[0]):
    fname, true_class = test_meta[i]
    single_img = test_tensor[i:i+1]

    # Load the heatmap we already saved in Cell 7 — no need to re-run SHAP
    heatmap_path = os.path.join(OUTPUT_DIR, true_class, f"{fname}_ensemble_shap.npy")
    if not os.path.exists(heatmap_path):
        print(f"WARNING: no saved heatmap found for {fname}, skipping")
        continue
    heatmap = np.load(heatmap_path)

    with torch.no_grad():
        probs = ensemble(single_img)
        pred_idx = torch.argmax(probs, dim=1).item()

    confidences = run_deletion_test(single_img, heatmap, ensemble, pred_idx, DEVICE)
    all_curves.append(confidences)

    # AOPC for this image: average confidence drop from the original (0% deleted)
    original_conf = confidences[0]
    aopc_this_image = np.mean([original_conf - c for c in confidences[1:]])

    per_class_aopc.setdefault(true_class, []).append(aopc_this_image)

    print(f"  [{i+1}/{test_tensor.shape[0]}] {fname} ({true_class}): AOPC = {aopc_this_image:.4f}")

    torch.cuda.empty_cache()

# ------------------------------------------------------------
# Overall AOPC score (single number, comparable to your individual
# model results: EfficientNet-B1 = 0.4990, ConvNeXt-Tiny = 0.3574)
# ------------------------------------------------------------
overall_aopc = np.mean([v for values in per_class_aopc.values() for v in values])
print(f"\n{'='*50}")
print(f"ENSEMBLE OVERALL AOPC: {overall_aopc:.4f}")
print(f"{'='*50}")

print("\nPer-class AOPC (checks if the class-imbalance finding persists):")
for class_name, values in per_class_aopc.items():
    print(f"  {class_name}: {np.mean(values):.4f}  (n={len(values)})")

# ------------------------------------------------------------
# Deletion curve plot — averaged across all test images
# ------------------------------------------------------------
mean_curve = np.mean(all_curves, axis=0)

plt.figure(figsize=(8, 6))
plt.plot(DELETION_STEPS, mean_curve, marker="o", linewidth=2, color="darkred")
plt.xlabel("% of pixels deleted (most important first)")
plt.ylabel("Confidence in original predicted class")
plt.title(f"Ensemble Deletion Curve (AOPC = {overall_aopc:.4f})")
plt.grid(alpha=0.3)
plt.tight_layout()
curve_path = os.path.join(AOPC_OUTPUT_DIR, "ensemble_deletion_curve.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.close()

# Save the raw numbers too, in case you need them for the paper's tables
np.save(os.path.join(AOPC_OUTPUT_DIR, "ensemble_deletion_curves_all.npy"), np.array(all_curves))
import json
with open(os.path.join(AOPC_OUTPUT_DIR, "ensemble_aopc_summary.json"), "w") as f:
    json.dump({
        "overall_aopc": float(overall_aopc),
        "per_class_aopc": {k: float(np.mean(v)) for k, v in per_class_aopc.items()},
    }, f, indent=2)

print(f"\nSaved deletion curve plot + raw data to: {AOPC_OUTPUT_DIR}")

  [1/15] ISIC_0065249.jpg (Basal_Cell_Carcinoma): AOPC = 0.6386
  [2/15] ISIC_0071603.jpg (Basal_Cell_Carcinoma): AOPC = 0.7378
  [3/15] ISIC_0073081.jpg (Basal_Cell_Carcinoma): AOPC = 0.6242
  [4/15] 6_29.jpg (Eczema): AOPC = 0.4085
  [5/15] IMG20240403112731.jpg (Eczema): AOPC = 0.2595
  [6/15] IMG_20230531_124822_1.jpg (Eczema): AOPC = 0.0457
  [7/15] ISIC_0062717.jpg (Keratosis): AOPC = 0.3845
  [8/15] ISIC_0071952.jpg (Keratosis): AOPC = 0.3462
  [9/15] v-epidermal-cyst-106.jpg (Keratosis): AOPC = 0.3405
  [10/15] 3635.jpg (Melanoma): AOPC = 0.7651
  [11/15] ISIC_6895778.jpg (Melanoma): AOPC = 0.4260
  [12/15] ISIC_7303761.jpg (Melanoma): AOPC = 0.3272
  [13/15] IMG_20230214_120109_1.jpg (Tinea): AOPC = 0.3344
  [14/15] IMG_20230705_122531.jpg (Tinea): AOPC = 0.2306
  [15/15] t-tinea-primary-lesion-14.jpg (Tinea): AOPC = 0.1723

ENSEMBLE OVERALL AOPC: 0.4027

Per-class AOPC (checks if the class-imbalance finding persists):
  Basal_Cell_Carcinoma: 0.6669  (n=3)
  Eczema: 0.2379  (n

In [ ]:
# ------------------------------------------------------------
# CELL 9 — Individual SHAP for all 3 models, WITH raw .npy arrays
# saved this time (your earlier EfficientNet/ConvNeXt runs only saved
# PNG images, not raw arrays — so we recompute all 3 here, consistently,
# to enable combining them in Cell 10). Reuses models already loaded
# inside `ensemble` — no need to reload checkpoints.
# ------------------------------------------------------------
import gc
import numpy as np
import shap
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

class SingleModelWrapper(nn.Module):
    """Wraps one backbone so SHAP sees it alone (softmax'd)."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return torch.softmax(self.model(x), dim=1)

INDIVIDUAL_MODELS = {
    "efficientnet_b1": ensemble.efficientnet,
    "convnext_tiny": ensemble.convnext,
    "swin_tiny": ensemble.swin,
}

# Swin's transformer-based attention tends to produce more diffuse/global
# explanations than the CNNs (a known ViT characteristic, not a bug).
# If you want to try tightening it, raise this value just for Swin —
# stronger smoothing can pull the explanation more toward the lesion.
# Leave at 0.5 to keep results consistent with your documented methodology.
LOCAL_SMOOTHING_PER_MODEL = {
    "efficientnet_b1": 0.5,
    "convnext_tiny": 0.5,
    "swin_tiny": 0.5,   # reverted from 0.8 — scored worse on AOPC despite looking visually cleaner
}

INDIVIDUAL_RESULTS_DIR = f"{DRIVE_ROOT}/SHAP/individual_results_with_arrays"
if os.path.exists(INDIVIDUAL_RESULTS_DIR):
    shutil.rmtree(INDIVIDUAL_RESULTS_DIR)
os.makedirs(INDIVIDUAL_RESULTS_DIR, exist_ok=True)
print("Individual results folder ready:", INDIVIDUAL_RESULTS_DIR)

for model_name, model in INDIVIDUAL_MODELS.items():
    print(f"\n--- Running SHAP for {model_name} ---")
    wrapped = SingleModelWrapper(model).to(DEVICE)
    wrapped.eval()

    model_output_dir = os.path.join(INDIVIDUAL_RESULTS_DIR, model_name)
    os.makedirs(model_output_dir, exist_ok=True)

    torch.cuda.empty_cache()
    gc.collect()

    explainer = shap.GradientExplainer(wrapped, background_tensor, local_smoothing=LOCAL_SMOOTHING_PER_MODEL[model_name], batch_size=1)

    for i in range(test_tensor.shape[0]):
        fname, true_class = test_meta[i]
        single_img = test_tensor[i:i+1]

        with torch.no_grad():
            probs = wrapped(single_img)
            pred_idx = torch.argmax(probs, dim=1).item()
            pred_class = ensemble.classes[pred_idx]
            pred_confidence = probs[0, pred_idx].item()

        result = explainer.shap_values(single_img, ranked_outputs=1, nsamples=150)

        if isinstance(result, tuple):
            shap_values, indexes = result
        else:
            shap_values, indexes = result, None

        if isinstance(shap_values, list):
            sv = shap_values[0]
        else:
            sv = shap_values

        sv = np.array(sv)
        sv = sv.reshape(sv.shape[1:4]) if sv.ndim == 5 else sv[0]

        # Unsigned heatmap (abs sum across channels) — consistent
        # across all 3 models so they can be directly combined later
        heatmap = np.sum(np.abs(sv), axis=0)
        heatmap_blurred = gaussian_filter(heatmap, sigma=3)

        original_img = unnormalize(single_img[0])

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(original_img)
        axes[0].set_title(f"True: {true_class}\nPred: {pred_class}", loc="left", fontsize=11)
        axes[0].axis("off")

        axes[1].imshow(heatmap_blurred, cmap="hot")
        axes[1].set_title(f"SHAP ({model_name})")
        axes[1].axis("off")

        axes[2].imshow(original_img)
        axes[2].imshow(heatmap_blurred, cmap="hot", alpha=0.5)
        axes[2].set_title("Overlay")
        axes[2].axis("off")

        plt.tight_layout()

        class_dir = os.path.join(model_output_dir, true_class)
        os.makedirs(class_dir, exist_ok=True)

        out_path = os.path.join(class_dir, f"{fname}_{model_name}_shap.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Save the RAW heatmap (unblurred, for accurate combining) and
        # this model's own confidence (needed for confidence-weighting)
        np.save(os.path.join(class_dir, f"{fname}_{model_name}_shap.npy"), heatmap)
        np.save(os.path.join(class_dir, f"{fname}_{model_name}_confidence.npy"), np.array(pred_confidence))

        print(f"  [{i+1}/{test_tensor.shape[0]}] {fname} ({true_class}): conf={pred_confidence:.3f}")

        del result, shap_values, sv, heatmap, heatmap_blurred, original_img, fig
        gc.collect()
        torch.cuda.empty_cache()

    del wrapped, explainer
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nDone! All 3 models' individual SHAP results (with raw arrays) saved to: {INDIVIDUAL_RESULTS_DIR}")

Individual results folder ready: /content/drive/MyDrive/IPD/XAI/SHAP/individual_results_with_arrays

--- Running SHAP for efficientnet_b1 ---
  [1/15] ISIC_0065249.jpg (Basal_Cell_Carcinoma): conf=0.948
  [2/15] ISIC_0071603.jpg (Basal_Cell_Carcinoma): conf=0.943
  [3/15] ISIC_0073081.jpg (Basal_Cell_Carcinoma): conf=0.972
  [4/15] 6_29.jpg (Eczema): conf=0.883
  [5/15] IMG20240403112731.jpg (Eczema): conf=0.864
  [6/15] IMG_20230531_124822_1.jpg (Eczema): conf=0.585
  [7/15] ISIC_0062717.jpg (Keratosis): conf=0.650
  [8/15] ISIC_0071952.jpg (Keratosis): conf=0.897
  [9/15] v-epidermal-cyst-106.jpg (Keratosis): conf=0.937
  [10/15] 3635.jpg (Melanoma): conf=0.953
  [11/15] ISIC_6895778.jpg (Melanoma): conf=0.876
  [12/15] ISIC_7303761.jpg (Melanoma): conf=0.803
  [13/15] IMG_20230214_120109_1.jpg (Tinea): conf=0.987
  [14/15] IMG_20230705_122531.jpg (Tinea): conf=0.982
  [15/15] t-tinea-primary-lesion-14.jpg (Tinea): conf=0.941

--- Running SHAP for convnext_tiny ---
  [1/15] ISIC_0065

In [ ]:
# ------------------------------------------------------------
# CELL 10 — Combine the 3 models' heatmaps, 3 different ways
# (this IS your paper's original contribution). For each test image,
# loads the 3 raw heatmaps + confidences from Cell 9 and produces:
#   1. Simple average
#   2. Confidence-weighted average
#   3. Agreement/voting (keep pixel only if ≥2 of 3 models call it
#      important, i.e. in that model's own top 20% most important pixels)
# Saves each strategy's overlays into its own folder.
# ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

MODEL_NAMES = ["efficientnet_b1", "convnext_tiny", "swin_tiny"]
AGREEMENT_TOP_PCT = 20  # "important" = top 20% of that model's own heatmap

COMBINATION_STRATEGIES = ["simple_average", "confidence_weighted", "agreement_voting"]

# One output folder per strategy, each with per-class subfolders,
# living alongside your other ensemble results
COMBO_DIRS = {
    strategy: f"{DRIVE_ROOT}/SHAP/SHAP_newfolder_ensemble_results/combo_{strategy}"
    for strategy in COMBINATION_STRATEGIES
}
for d in COMBO_DIRS.values():
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
print("Combination output folders ready:", list(COMBO_DIRS.values()))


def binarize_top_pct(heatmap, pct):
    """Returns a 0/1 mask marking this model's own top `pct`% most
    important pixels — used for the agreement/voting strategy."""
    threshold = np.percentile(heatmap, 100 - pct)
    return (heatmap >= threshold).astype(float)


# Store combined heatmaps in memory too, for Cell 11's AOPC test
combined_heatmaps = {strategy: {} for strategy in COMBINATION_STRATEGIES}

for i in range(test_tensor.shape[0]):
    fname, true_class = test_meta[i]

    # Load each model's raw heatmap + confidence for this image
    heatmaps = {}
    confidences = {}
    missing = False
    for model_name in MODEL_NAMES:
        hpath = os.path.join(INDIVIDUAL_RESULTS_DIR, model_name, true_class, f"{fname}_{model_name}_shap.npy")
        cpath = os.path.join(INDIVIDUAL_RESULTS_DIR, model_name, true_class, f"{fname}_{model_name}_confidence.npy")
        if not os.path.exists(hpath) or not os.path.exists(cpath):
            print(f"WARNING: missing {model_name} data for {fname}, skipping this image")
            missing = True
            break
        heatmaps[model_name] = np.load(hpath)
        confidences[model_name] = float(np.load(cpath))
    if missing:
        continue

    # Normalize each model's heatmap to [0,1] before combining, so no
    # single model dominates just because its raw SHAP scale is bigger.
    # Uses the 99th percentile instead of the raw max — this avoids one
    # single noisy/outlier pixel (e.g. on the scope border) from
    # dominating the whole combined heatmap.
    norm_heatmaps = {}
    for model_name, hmap in heatmaps.items():
        h99 = np.percentile(hmap, 99)
        norm = hmap / h99 if h99 > 0 else hmap
        norm_heatmaps[model_name] = np.clip(norm, 0, 1)

    # --- Strategy 1: Simple average ---
    simple_avg = np.mean([norm_heatmaps[m] for m in MODEL_NAMES], axis=0)

    # --- Strategy 2: Confidence-weighted average ---
    total_conf = sum(confidences.values())
    if total_conf > 0:
        weighted_avg = sum(norm_heatmaps[m] * confidences[m] for m in MODEL_NAMES) / total_conf
    else:
        weighted_avg = simple_avg.copy()

    # --- Strategy 3: Agreement/voting ---
    binary_masks = [binarize_top_pct(norm_heatmaps[m], AGREEMENT_TOP_PCT) for m in MODEL_NAMES]
    agreement_count = np.sum(binary_masks, axis=0)  # 0, 1, 2, or 3
    agreement_keep = (agreement_count >= 2).astype(float)
    # Where ≥2 models agree, use their average magnitude; elsewhere zero
    magnitude_avg = np.mean([norm_heatmaps[m] for m in MODEL_NAMES], axis=0)
    voting_result = magnitude_avg * agreement_keep

    strategy_results = {
        "simple_average": simple_avg,
        "confidence_weighted": weighted_avg,
        "agreement_voting": voting_result,
    }

    # Get the ensemble's actual prediction for the title (same as Cell 7/8)
    single_img = test_tensor[i:i+1]
    with torch.no_grad():
        probs = ensemble(single_img)
        pred_idx = torch.argmax(probs, dim=1).item()
        pred_class = ensemble.classes[pred_idx]
    original_img = unnormalize(single_img[0])

    for strategy, combined_map in strategy_results.items():
        combined_heatmaps[strategy][(true_class, fname)] = combined_map

        blurred = gaussian_filter(combined_map, sigma=3)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(original_img)
        axes[0].set_title(f"True: {true_class}\nPred: {pred_class}", loc="left", fontsize=11)
        axes[0].axis("off")

        axes[1].imshow(blurred, cmap="hot")
        axes[1].set_title(f"Combined ({strategy})")
        axes[1].axis("off")

        axes[2].imshow(original_img)
        axes[2].imshow(blurred, cmap="hot", alpha=0.5)
        axes[2].set_title("Overlay")
        axes[2].axis("off")

        plt.tight_layout()

        class_dir = os.path.join(COMBO_DIRS[strategy], true_class)
        os.makedirs(class_dir, exist_ok=True)
        out_path = os.path.join(class_dir, f"{fname}_{strategy}.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        np.save(os.path.join(class_dir, f"{fname}_{strategy}.npy"), combined_map)

    print(f"[{i+1}/{test_tensor.shape[0]}] Combined all 3 strategies for {fname} ({true_class})")

print("\nDone! All 3 combination strategies computed and saved.")

Combination output folders ready: ['/content/drive/MyDrive/IPD/XAI/SHAP/SHAP_newfolder_ensemble_results/combo_simple_average', '/content/drive/MyDrive/IPD/XAI/SHAP/SHAP_newfolder_ensemble_results/combo_confidence_weighted', '/content/drive/MyDrive/IPD/XAI/SHAP/SHAP_newfolder_ensemble_results/combo_agreement_voting']
[1/15] Combined all 3 strategies for ISIC_0065249.jpg (Basal_Cell_Carcinoma)
[2/15] Combined all 3 strategies for ISIC_0071603.jpg (Basal_Cell_Carcinoma)
[3/15] Combined all 3 strategies for ISIC_0073081.jpg (Basal_Cell_Carcinoma)
[4/15] Combined all 3 strategies for 6_29.jpg (Eczema)
[5/15] Combined all 3 strategies for IMG20240403112731.jpg (Eczema)
[6/15] Combined all 3 strategies for IMG_20230531_124822_1.jpg (Eczema)
[7/15] Combined all 3 strategies for ISIC_0062717.jpg (Keratosis)
[8/15] Combined all 3 strategies for ISIC_0071952.jpg (Keratosis)
[9/15] Combined all 3 strategies for v-epidermal-cyst-106.jpg (Keratosis)
[10/15] Combined all 3 strategies for 3635.jpg (Me

In [ ]:
# ------------------------------------------------------------
# CELL 11 — AOPC deletion test for all 3 combination strategies
# (same methodology as Cell 8, just run 3 more times — one per
# strategy). Confidence is measured using the ENSEMBLE's own
# prediction, since these heatmaps are meant to explain the
# ensemble's decision, just built by combining individual explanations.
# ------------------------------------------------------------
import json

AOPC_COMBO_DIR = f"{DRIVE_ROOT}/SHAP/SHAP_newfolder_ensemble_results/AOPC_results_combinations"
os.makedirs(AOPC_COMBO_DIR, exist_ok=True)

all_strategy_results = {}

for strategy in COMBINATION_STRATEGIES:
    print(f"\n=== Running AOPC for strategy: {strategy} ===")
    all_curves = []
    per_class_aopc = {}

    for i in range(test_tensor.shape[0]):
        fname, true_class = test_meta[i]
        key = (true_class, fname)
        if key not in combined_heatmaps[strategy]:
            continue

        heatmap = combined_heatmaps[strategy][key]
        single_img = test_tensor[i:i+1]

        with torch.no_grad():
            probs = ensemble(single_img)
            pred_idx = torch.argmax(probs, dim=1).item()

        confidences = run_deletion_test(single_img, heatmap, ensemble, pred_idx, DEVICE)
        all_curves.append(confidences)

        original_conf = confidences[0]
        aopc_this_image = np.mean([original_conf - c for c in confidences[1:]])
        per_class_aopc.setdefault(true_class, []).append(aopc_this_image)

        torch.cuda.empty_cache()

    overall_aopc = np.mean([v for values in per_class_aopc.values() for v in values])
    mean_curve = np.mean(all_curves, axis=0)

    print(f"{strategy} — Overall AOPC: {overall_aopc:.4f}")
    for class_name, values in per_class_aopc.items():
        print(f"    {class_name}: {np.mean(values):.4f}")

    # Save this strategy's deletion curve plot
    plt.figure(figsize=(8, 6))
    plt.plot(DELETION_STEPS, mean_curve, marker="o", linewidth=2, color="darkblue")
    plt.xlabel("% of pixels deleted (most important first)")
    plt.ylabel("Confidence in ensemble's predicted class")
    plt.title(f"{strategy} Deletion Curve (AOPC = {overall_aopc:.4f})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(AOPC_COMBO_DIR, f"{strategy}_deletion_curve.png"), dpi=150, bbox_inches="tight")
    plt.close()

    all_strategy_results[strategy] = {
        "overall_aopc": float(overall_aopc),
        "per_class_aopc": {k: float(np.mean(v)) for k, v in per_class_aopc.items()},
    }

# ------------------------------------------------------------
# FINAL COMPARISON — this answers your actual research question:
# "what's the best way to combine explanations from a multi-architecture
# ensemble?" Compares all 4 approaches side by side.
# ------------------------------------------------------------
all_strategy_results["wrapped_ensemble_shap"] = {"overall_aopc": float(wrapped_ensemble_aopc) if 'wrapped_ensemble_aopc' in dir() else "run Cell 8 first"}

print("\n" + "="*60)
print("FINAL COMPARISON — AOPC scores (higher = more faithful)")
print("="*60)
for strategy, results in all_strategy_results.items():
    print(f"  {strategy}: {results['overall_aopc']}")
print("="*60)
print("Whichever score is HIGHEST is your recommended method.")

with open(os.path.join(AOPC_COMBO_DIR, "all_strategies_comparison.json"), "w") as f:
    json.dump(all_strategy_results, f, indent=2)

print(f"\nSaved full comparison to: {os.path.join(AOPC_COMBO_DIR, 'all_strategies_comparison.json')}")



=== Running AOPC for strategy: simple_average ===
simple_average — Overall AOPC: 0.3829
    Basal_Cell_Carcinoma: 0.6792
    Eczema: 0.2868
    Keratosis: 0.3317
    Melanoma: 0.4395
    Tinea: 0.1773

=== Running AOPC for strategy: confidence_weighted ===
confidence_weighted — Overall AOPC: 0.3770
    Basal_Cell_Carcinoma: 0.6854
    Eczema: 0.2793
    Keratosis: 0.3167
    Melanoma: 0.4327
    Tinea: 0.1708

=== Running AOPC for strategy: agreement_voting ===
agreement_voting — Overall AOPC: 0.2787
    Basal_Cell_Carcinoma: 0.5490
    Eczema: 0.2046
    Keratosis: 0.1786
    Melanoma: 0.3006
    Tinea: 0.1606

FINAL COMPARISON — AOPC scores (higher = more faithful)
  simple_average: 0.382898696983854
  confidence_weighted: 0.37697601735591885
  agreement_voting: 0.27867345766474805
  wrapped_ensemble_shap: run Cell 8 first
Whichever score is HIGHEST is your recommended method.

Saved full comparison to: /content/drive/MyDrive/IPD/XAI/SHAP/SHAP_newfolder_ensemble_results/AOPC_results

In [ ]:
# ------------------------------------------------------------
# FOLDER ORGANIZATION — run this after Cell 0 (Drive mount).
# Renames folders in place with clear numbered prefixes so it's
# obvious at a glance which results are old vs. final/current.
# Does NOT delete anything — pure rename, fully reversible.
# ------------------------------------------------------------
import os

SHAP_ROOT = "/content/drive/MyDrive/IPD/XAI/SHAP"

RENAMES = {
    "background": "00_background_images_INPUT",
    "test": "00_test_images_INPUT",
    "shap_outputs": "01_EARLIER_individual_SHAP_EfficientNet_ConvNeXt_PNGonly_NO_ARRAYS",
    "individual_results_with_arrays": "02_FINAL_individual_SHAP_all3models_with_arrays",
    "SHAP_newfolder_ensemble_results": "03_FINAL_ensemble_XAI_results_wrapped_and_combinations",
}

for old_name, new_name in RENAMES.items():
    old_path = os.path.join(SHAP_ROOT, old_name)
    new_path = os.path.join(SHAP_ROOT, new_name)
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
        print(f"Renamed: {old_name}  ->  {new_name}")
    elif os.path.exists(new_path):
        print(f"Already renamed: {new_name}")
    else:
        print(f"WARNING: could not find '{old_name}' — check the name/path manually")

print("\nFinal structure:")
for item in sorted(os.listdir(SHAP_ROOT)):
    print(" ", item)

Renamed: background  ->  00_background_images_INPUT
Renamed: test  ->  00_test_images_INPUT
Renamed: shap_outputs  ->  01_EARLIER_individual_SHAP_EfficientNet_ConvNeXt_PNGonly_NO_ARRAYS
Renamed: individual_results_with_arrays  ->  02_FINAL_individual_SHAP_all3models_with_arrays
Renamed: SHAP_newfolder_ensemble_results  ->  03_FINAL_ensemble_XAI_results_wrapped_and_combinations

Final structure:
  00_background_images_INPUT
  00_test_images_INPUT
  01_EARLIER_individual_SHAP_EfficientNet_ConvNeXt_PNGonly_NO_ARRAYS
  02_FINAL_individual_SHAP_all3models_with_arrays
  03_FINAL_ensemble_XAI_results_wrapped_and_combinations
